# U5_05 — RAG, Memoria Avanzada y APIs de Datos

> **Unidad:** 5 · Sistemas Multi-Agente Modernos  
> **Dificultad:** Intermedia-Avanzada ★★★★☆  
> **Duración estimada:** 6–7 horas  
> **Prereq. mínimo:** U5_02 (LangChain + LangGraph)  
> **Skills que se crean:** `episodic_retriever`, `token_budget_guard`  
> **SCOPE:** Esta notebook cubre memoria vectorial y scalar. Grafos de conocimiento (Neo4j, Graphiti, GraphRAG) son exclusivos de U5_06.

---

## Tabla de Contenidos

1. [Objetivos de Aprendizaje](#1-objetivos)
2. [Warm-Up del Entorno](#2-warmup)
3. [Taxonomía Formal de Memoria en Agentes](#3-taxonomia)
   - 3.1 Los 4 Tipos de Memoria
   - 3.2 Coordinación de los 4 Tipos en un Agente
   - 3.3 La Analogía del Ser Humano
4. [Arquitecturas RAG](#4-rag)
   - 4.1 Naive RAG vs Advanced RAG vs Agentic RAG
   - 4.2 Estrategias de Chunking
   - 4.3 Embeddings y Modelos de Representación
   - 4.4 Reranking: Por Qué un Solo Retriever No Es Suficiente
   - 4.5 Implementación: Pipeline RAG Completo con LangChain
5. [Memoria Episódica Vectorial](#5-episodica)
   - 5.1 Mem0: Eventos como Vectores con Timestamps
   - 5.2 Skill `episodic_retriever`: Warm-Up con Episodios Pasados
   - 5.3 Long-Term Memory Semántica con ChromaDB
6. [Optimización de Tokens y Presupuesto](#6-tokens)
   - 6.1 El Problema: Costo = Riesgo Real
   - 6.2 Estrategias: Summarization, Selective Loading, Semantic Caching
   - 6.3 Skill `token_budget_guard`: Circuit Breaker en Tiempo Real
7. [APIs de Datos como Fuentes de Memoria Semántica](#7-apis)
   - 7.1 arXiv: Literatura Científica en Tiempo Real
   - 7.2 PubMed: Biomedicina y Nanomedicina
   - 7.3 Alpha Vantage: Datos Financieros
   - 7.4 Patrón General: Cualquier API como Fuente RAG
8. [Proyecto: Sistema de Investigación con los 4 Tipos de Memoria](#8-proyecto)
9. [Resumen y Conexión con U5_06 y U5_07](#9-resumen)
10. [Ejercicios de Extensión](#10-ejercicios)

---
<a id='1-objetivos'></a>
## 1. Objetivos de Aprendizaje

Al completar esta notebook el estudiante será capaz de:

**CORE (evaluables):**
- Distinguir los 4 tipos de memoria y asignar una tecnología de implementación a cada uno
- Implementar un pipeline RAG con chunking semántico, embeddings y reranking
- Integrar Mem0 para memoria episódica que persiste entre sesiones
- Construir la skill `token_budget_guard` y demostrar que interrumpe el flujo al superar el presupuesto
- Conectar una API externa (arXiv o PubMed) como fuente dinámica de un vector store

**COMPLEMENTARIOS:**
- Implementar semantic caching para reducir llamadas redundantes al LLM
- Comparar latencia y costo entre chunking fixed vs semantic
- Combinar memoria episódica + semántica en un único agente de investigación

> **Criterio de evaluación:** Construir el sistema de investigación de la Sección 8 con los 4 tipos de memoria activos y el `token_budget_guard` configurado para un presupuesto de $0.05.

---
<a id='2-warmup'></a>
## 2. Warm-Up del Entorno

In [1]:
import subprocess, sys, os
from pathlib import Path

def check_install(package, import_name=None):
    name = import_name or package.split('==')[0].replace('-', '_')
    try:
        __import__(name)
        print(f'  OK: {package}')
    except ImportError:
        print(f'  Instalando {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

# Paso 1: Verificar dependencias de esta notebook
packages = [
    ('langchain==0.3.25', 'langchain'),
    ('langchain-community==0.3.24', 'langchain_community'),
    ('langchain-openai==0.3.12', 'langchain_openai'),
    ('langchain-google-genai==2.1.4', 'langchain_google_genai'),
    ('langchain-ollama==0.3.2', 'langchain_ollama'),
    ('chromadb==0.6.3', 'chromadb'),
    ('sentence-transformers==3.4.1', 'sentence_transformers'),
    ('mem0ai==0.1.103', 'mem0'),
    ('tiktoken==0.9.0', 'tiktoken'),
    ('httpx==0.28.1', 'httpx'),
    ('python-dotenv==1.1.0', 'dotenv'),
]
for pkg, imp in packages:
    check_install(pkg, imp)

# Paso 2: Cargar variables de entorno
from dotenv import load_dotenv
import os
load_dotenv()

GOOGLE_API_KEY     = os.environ.get('GOOGLE_API_KEY')
OPENAI_API_KEY     = os.environ.get('OPENAI_API_KEY')    # opcional
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY') # opcional
MEM0_API_KEY       = os.environ.get('MEM0_API_KEY')       # opcional (con fallback local)

if not any([OPENROUTER_API_KEY, GOOGLE_API_KEY, OPENAI_API_KEY]):
    print('\n[AVISO] No se encontró OPENROUTER_API_KEY, GOOGLE_API_KEY ni OPENAI_API_KEY.')
    print('Usando Ollama (llama3.2) como modelo local. Asegúrate de tener ollama serve corriendo.')
    USE_LOCAL = True
else:
    USE_LOCAL = False

backend = (
    "OpenRouter"     if OPENROUTER_API_KEY else
    "Gemini"         if GOOGLE_API_KEY     else
    # "OpenAI"         if OPENAI_API_KEY     else
    # "Ollama (local)"
    "Sin backend — configura OPENROUTER_API_KEY o GOOGLE_API_KEY en .env"
)

print('\nWarm-up completado.')
print(f'  Modelo: {backend}')
print(f'  Mem0 cloud: {"activo" if MEM0_API_KEY else "no configurado — usando fallback local"}')


  OK: langchain==0.3.25
  OK: langchain-community==0.3.24
  OK: langchain-openai==0.3.12
  OK: langchain-google-genai==2.1.4
  OK: langchain-ollama==0.3.2
  OK: chromadb==0.6.3

  OK: sentence-transformers==3.4.1
  OK: mem0ai==0.1.103
  OK: tiktoken==0.9.0
  OK: httpx==0.28.1
  OK: python-dotenv==1.1.0

Warm-up completado.
  Modelo: OpenRouter
  Mem0 cloud: activo


---
<a id='3-taxonomia'></a>
## 3. Taxonomía Formal de Memoria en Agentes

### 3.1 Los 4 Tipos de Memoria

La memoria en sistemas de IA es uno de los temas más malentendidos del campo. Cuando los desarrolladores hablan de "darle memoria a un agente", a menudo se refieren a una sola cosa: pasar los mensajes anteriores en el contexto. Pero eso es solo uno de cuatro tipos de memoria fundamentalmente distintos, cada uno con características, tecnologías de implementación, y propósitos diferentes.

Esta taxonomía proviene de la convergencia de tres fuentes: la literatura cognitiva sobre memoria humana (Tulving, 1985), el framework MemGPT/Letta (Packer et al., 2023), y los patrones de producción de Mem0 y Zep (2024-2026).

| Tipo | ¿Qué almacena? | Duración | Implementación | Analogía humana |
|------|----------------|----------|----------------|-----------------|
| **Sensorial / In-Context** | El historial activo de la conversación actual, contexto inyectado en el prompt | Solo mientras el token está en la ventana | `ConversationBufferMemory`, mensajes en el `state` de LangGraph | Memoria de trabajo (RAM) |
| **Episódica** | Eventos y experiencias pasadas: "la sesión del martes", "el usuario pidió X hace 3 días" | Persistente entre sesiones | Mem0, Zep, tablas de episodios en PostgreSQL, MemGPT/Letta | Memoria autobiográfica |
| **Semántica** | Conocimiento factual y conceptual: documentos, FAQs, bases de conocimiento | Persistente, actualizable | ChromaDB, Pinecone, Weaviate, pgvector + RAG | Memoria declarativa / conocimiento general |
| **Procedimental** | Cómo hacer cosas: skills, herramientas, estrategias de razonamiento | Persistente, versionada | Skill Registry, few-shot examples, fine-tuning, LoRA | Memoria muscular / hábitos |

#### ¿Por qué importa esta distinción?

Un agente que solo tiene memoria **sensorial** (buffer del historial) olvida todo al cerrar la sesión. Un agente con memoria **episódica** recuerda las preferencias del usuario de la semana pasada. Un agente con memoria **semántica** puede responder preguntas sobre un documento que nunca estuvo en su contexto. Un agente con memoria **procedimental** puede "aprender" nuevas capacidades sin fine-tuning.

La diferencia entre un chatbot básico y un asistente persistente productivo es exactamente cuántos de estos tipos implementa.

### 3.2 Coordinación de los 4 Tipos en un Agente

El patrón de coordinación más efectivo es el **warm-up secuencial**: antes de responder a la primera consulta de una sesión, el agente carga activamente los tipos de memoria que necesita:

```
INICIO DE SESIÓN
     ↓
1. [Procedimental] Cargar skills del registry → define qué PUEDE hacer el agente
2. [Episódica]     Recuperar episodios pasados relevantes → define lo que RECUERDA
3. [Semántica]     Inicializar vector store listo para queries → define lo que SABE
4. [Sensorial]     Contexto activo vacío, listo para recibir la primera consulta
     ↓
PRIMERA CONSULTA DEL USUARIO
     ↓
[Semántica] RAG retrieval → chunks relevantes al query
[Episódica] episodic_retriever → episodios pasados relevantes al query
[Sensorial] Historial de la sesión actual
[Procedimental] Skill apropiada según la tarea (routing)
     ↓
LLM con contexto enriquecido → RESPUESTA
     ↓
FIN DE SESIÓN → guardar nuevo episodio en memoria episódica
```

### 3.3 La Analogía del Ser Humano

Para anclar estos conceptos en intuición:

- **Sensorial**: Lo que tienes en la mente ahora mismo leyendo esta oración. Cuando termines de leer este párrafo, los detalles exactos se difuminarán.
- **Episódica**: Recuerdas lo que desayunaste el lunes pasado, quién te lo preparó, y la conversación que tuviste. Es autobiográfica, tiene contexto temporal.
- **Semántica**: Sabes que el agua hierve a 100°C a nivel del mar. No recuerdas dónde aprendiste eso — simplemente lo sabes. Es conocimiento descontextualizado.
- **Procedimental**: Sabes montar en bicicleta. No puedes explicar exactamente qué músculos activas — lo haces automáticamente. Las skills del agente son esto: capacidades codificadas que se activan.

La diferencia entre memoria semántica y episódica es sutil pero crucial en ingeniería: la episódica tiene **quién, cuándo, dónde** (metadatos temporales y contextuales); la semántica solo tiene **qué** (contenido factual).

In [2]:
# ============================================================
# Demostración: Los 4 tipos de memoria en código
# ============================================================
# Paso 1: Importar clases de LangChain para memoria in-context
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain_core.messages import HumanMessage, AIMessage

# Paso 2: MEMORIA SENSORIAL — buffer del historial activo
sensorial_memory = ConversationBufferMemory(return_messages=True)
sensorial_memory.save_context(
    {"input": "¿Cuál es la capital de México?"},
    {"output": "La capital de México es Ciudad de México."}
)
sensorial_memory.save_context(
    {"input": "¿Y su población aproximada?"},
    {"output": "Ciudad de México tiene aproximadamente 9 millones de habitantes en la ciudad central."}
)

print('=== MEMORIA SENSORIAL (Buffer activo) ===')
print(f'Mensajes en contexto: {len(sensorial_memory.chat_memory.messages)}')
for msg in sensorial_memory.chat_memory.messages:
    role = 'Human' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}]: {msg.content[:60]}...')

# Paso 3: Mostrar cómo la memoria semántica (RAG) complementa lo anterior
print('\n=== DIFERENCIA CLAVE ===')
print('Sensorial: retiene el HILO de la conversación activa')
print('Episódica: retiene EVENTOS de conversaciones anteriores')
print('Semántica: retiene CONOCIMIENTO factual de documentos')
print('Procedimental: retiene CAPACIDADES (skills) del agente')

=== MEMORIA SENSORIAL (Buffer activo) ===
Mensajes en contexto: 4
  [Human]: ¿Cuál es la capital de México?...
  [AI]: La capital de México es Ciudad de México....
  [Human]: ¿Y su población aproximada?...
  [AI]: Ciudad de México tiene aproximadamente 9 millones de habitan...

=== DIFERENCIA CLAVE ===
Sensorial: retiene el HILO de la conversación activa
Episódica: retiene EVENTOS de conversaciones anteriores
Semántica: retiene CONOCIMIENTO factual de documentos
Procedimental: retiene CAPACIDADES (skills) del agente


C:\Users\UCEMICH\AppData\Local\Temp\ipykernel_14696\1504865000.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  sensorial_memory = ConversationBufferMemory(return_messages=True)


**Output esperado:**
```
=== MEMORIA SENSORIAL (Buffer activo) ===
Mensajes en contexto: 4
  [Human]: ¿Cuál es la capital de México?...
  [AI]: La capital de México es Ciudad de México....
  [Human]: ¿Y su población aproximada?...
  [AI]: Ciudad de México tiene aproximadamente 9 mi...
```

**Interpretación:** La memoria sensorial contiene 4 mensajes (2 pares Human/AI). Al cerrar esta sesión, estos mensajes desaparecerán. Para conservar que "el usuario preguntó sobre Ciudad de México", necesitamos memoria episódica. Para que el agente sepa datos actualizados sobre la ciudad, necesitamos memoria semántica con RAG.

---
<a id='4-rag'></a>
## 4. Arquitecturas RAG

### 4.1 Naive RAG vs Advanced RAG vs Agentic RAG

RAG (Retrieval-Augmented Generation) es el patrón de memoria semántica más usado en producción. En esencia: en lugar de hacer fine-tuning del modelo con datos propios, se recuperan los datos relevantes en tiempo de inferencia y se inyectan en el contexto.

Pero no todo RAG es igual. Existe una progresión de complejidad con trade-offs claros:

```
NAIVE RAG (2022-2023)
  Query → embed → k-NN → chunks → LLM
  Problema: chunks irrelevantes degrades la respuesta
  Problema: sin feedback loop — no aprende de malos resultados

ADVANCED RAG (2023-2024)
  Query → rewrite → embed → k-NN → rerank → filter → LLM
  Mejora: query rewriting mejora la recuperación
  Mejora: reranking filtra falsos positivos semánticos
  Problema: pipeline fijo — no puede decidir cuándo recuperar

AGENTIC RAG (2024-2026) ← lo que usamos en esta notebook
  El agente decide:
    - Si necesita recuperar o ya sabe la respuesta
    - Qué query enviar al retriever (puede reformularla)
    - Si el resultado es suficiente o necesita más iteraciones
    - De qué fuentes recuperar (múltiples vector stores)
  Implementación: LangGraph nodo de retrieval + condición de suficiencia
```

**¿Cuándo usar cada uno?**

| Escenario | RAG recomendado |
|-----------|----------------|
| FAQ estático, bajo tráfico | Naive RAG |
| Documentos técnicos con terminología específica | Advanced RAG + reranking |
| Sistema que necesita decidir cuándo buscar | Agentic RAG |
| Corpus masivo (>100k docs) con preguntas complejas | GraphRAG (ver U5_06) |

### 4.2 Estrategias de Chunking

El chunking es el paso más subestimado de RAG. La calidad del chunking determina en gran medida la calidad de la recuperación, más que el modelo de embeddings.

**El problema de chunk_size fijo:** Un chunk de 500 tokens que corta en medio de un párrafo técnico pierde el contexto. Un chunk demasiado grande incluye información irrelevante que confunde al reranker.

Estrategias en orden de sofisticación:

1. **Fixed-size**: `RecursiveCharacterTextSplitter(chunk_size=500)` — rápido, funciona para texto homogéneo
2. **Recursive con separadores**: respeta párrafos, oraciones, palabras en ese orden — mejor para prosa técnica
3. **Semantic chunking**: agrupa oraciones por similitud semántica — mejor para documentos con secciones heterogéneas
4. **Agentic chunking**: el LLM decide los límites del chunk — máxima calidad, máximo costo

### 4.3 Embeddings y Modelos de Representación

Los embeddings convierten texto en vectores densos. La elección del modelo afecta directamente la calidad del retrieval.

| Modelo | Dimensiones | Multilingüe | Costo | Mejor para |
|--------|-------------|-------------|-------|------------|
| `text-embedding-3-large` (OpenAI) | 3072 | Sí | API | Inglés técnico, alto rendimiento |
| `text-embedding-3-small` (OpenAI) | 1536 | Sí | Bajo | Producción, balance costo/calidad |
| `BGE-M3` (BAAI) | 1024 | Sí (100+ idiomas) | Local | Multilingüe, español |
| `nomic-embed-text` (Nomic) | 768 | No | Local/API | Documentos largos, contexto 8192 tokens |
| `sentence-transformers/all-MiniLM-L6-v2` | 384 | No | Local, rápido | Prototipado, bajo costo computacional |

### 4.4 Reranking: Por Qué un Solo Retriever No Es Suficiente

El retrieval con embeddings (k-NN) usa **similitud semántica aproximada**. El reranker usa un **cross-encoder** más lento pero preciso que evalúa el par (query, chunk) completo.

```
k-NN retrieval → top-50 candidatos (rápido, ~ms)
      ↓
cross-encoder reranker → top-5 mejores (lento, ~200ms, pero mucho más preciso)
      ↓
LLM con los 5 mejores chunks
```

El reranking mejora la precisión entre 15-40% en benchmarks de QA comparado con k-NN solo (Nogueira et al., 2020; Cohere Rerank benchmarks, 2024).

### 4.5 Implementación: Pipeline RAG Completo con LangChain

In [3]:
# ============================================================
# WARM-UP: RAG Pipeline
# ============================================================
import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

print('Importaciones RAG cargadas correctamente.')

Importaciones RAG cargadas correctamente.


In [4]:
# ============================================================
# Paso 1: Preparar documentos de ejemplo
# ============================================================
# Usamos documentos sobre sistemas multi-agente (dominio del curso)
docs_raw = [
    Document(
        page_content="""LangGraph es un framework para construir aplicaciones multi-agente con estado.
        Extiende LangChain añadiendo capacidades de grafos: nodos (funciones), aristas (transiciones),
        y estado compartido. Soporta ciclos, condicionales, y checkpointing. Ideal para workflows
        complejos donde los agentes necesitan tomar decisiones iterativas.""",
        metadata={"source": "langchain_docs", "topic": "frameworks", "date": "2026-01"}
    ),
    Document(
        page_content="""CrewAI es un framework role-based para sistemas multi-agente.
        Cada agente tiene un rol, objetivo, y backstory. Las tareas se asignan a agentes específicos
        o se distribuyen automáticamente. Soporta procesos secuenciales y jerárquicos. CrewAI Flow
        permite combinar crews con código Python arbitrario para mayor flexibilidad.""",
        metadata={"source": "crewai_docs", "topic": "frameworks", "date": "2026-01"}
    ),
    Document(
        page_content="""La memoria episódica en agentes de IA permite retener eventos de sesiones
        anteriores. A diferencia de la memoria semántica (conocimiento factual), la memoria
        episódica tiene contexto temporal: qué ocurrió, cuándo, y en qué conversación.
        Implementaciones: Mem0, Zep, MemGPT/Letta, tablas de episodios en PostgreSQL.""",
        metadata={"source": "memoir_paper", "topic": "memory", "date": "2024-12"}
    ),
    Document(
        page_content="""RAG (Retrieval-Augmented Generation) combina un retriever de similitud
        semántica con un generador LLM. El retriever busca en un vector store los chunks
        más relevantes para la query, y el LLM genera la respuesta usando esos chunks como contexto.
        Ventaja principal: el LLM puede responder preguntas sobre documentos que no estaban
        en sus datos de entrenamiento, sin fine-tuning.""",
        metadata={"source": "rag_survey", "topic": "rag", "date": "2024-03"}
    ),
    Document(
        page_content="""El protocolo MCP (Model Context Protocol) de Anthropic estandariza
        la comunicación entre LLMs y herramientas externas. Define tres primitivas:
        Tools (funciones que el modelo puede llamar), Resources (datos que el modelo puede leer),
        y Prompts (plantillas reutilizables). MCP permite que cualquier LLM use cualquier
        herramienta compatible sin integración ad-hoc.""",
        metadata={"source": "mcp_spec", "topic": "protocols", "date": "2024-11"}
    ),
]

print(f'Documentos preparados: {len(docs_raw)}')
print(f'Temas: {set(d.metadata["topic"] for d in docs_raw)}')

Documentos preparados: 5
Temas: {'rag', 'memory', 'protocols', 'frameworks'}


In [5]:
# ============================================================
# Paso 2: Chunking con RecursiveCharacterTextSplitter
# ============================================================
# Configuramos separadores en orden de preferencia:
# 1. Doble salto de línea (párrafos)
# 2. Punto y seguido (oraciones)
# 3. Coma o espacio (como último recurso)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,          # max caracteres por chunk
    chunk_overlap=50,        # solapamiento para no perder contexto en el corte
    separators=['\n\n', '.', ',', ' '],
    length_function=len,
)

chunks = splitter.split_documents(docs_raw)

print(f'Documentos originales: {len(docs_raw)}')
print(f'Chunks generados: {len(chunks)}')
print(f'Longitud promedio de chunk: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars')
print(f'\nEjemplo — Chunk 1:')
print(f'  Contenido: "{chunks[0].page_content[:120]}..."')
print(f'  Metadata: {chunks[0].metadata}')

Documentos originales: 5
Chunks generados: 10
Longitud promedio de chunk: 193 chars

Ejemplo — Chunk 1:
  Contenido: "LangGraph es un framework para construir aplicaciones multi-agente con estado.
        Extiende LangChain añadiendo capa..."
  Metadata: {'source': 'langchain_docs', 'topic': 'frameworks', 'date': '2026-01'}


In [6]:
# ============================================================
# Paso 3: Embeddings con sentence-transformers (local, sin API)
# ============================================================
# Usamos all-MiniLM-L6-v2: rápido, 384 dims, ideal para prototipos
# Para producción: cambiar a 'BAAI/bge-m3' (multilingüe, 1024 dims)
print('Cargando modelo de embeddings (primera vez: ~80MB de descarga)...')

embeddings_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

# Alternativa multilingüe con mejor rendimiento:
# embeddings_model = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')

# Verificar dimensiones
test_embedding = embeddings_model.embed_query('test')
print(f'Modelo cargado. Dimensiones del embedding: {len(test_embedding)}')

Cargando modelo de embeddings (primera vez: ~80MB de descarga)...


C:\Users\UCEMICH\AppData\Local\Temp\ipykernel_14696\234019510.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(


Modelo cargado. Dimensiones del embedding: 384


In [7]:
# ============================================================
# Paso 4: Crear vector store ChromaDB en memoria
# ============================================================
import shutil, gc, time

def _close_and_rmtree(path, retries=4, delay=0.5):
    """Libera handles de ChromaDB y borra el directorio (compatible con Windows)."""
    global vectorstore
    # Liberar referencia sin llamar _system.stop() — evita corrupción del sistema
    if 'vectorstore' in globals() and vectorstore is not None:
        vectorstore = None
    gc.collect()
    gc.collect()
    for i in range(retries):
        try:
            shutil.rmtree(path)
            return True
        except PermissionError:
            time.sleep(delay)
    return False

CHROMA_PATH = './chroma_u5_05'
if os.path.exists(CHROMA_PATH):
    if _close_and_rmtree(CHROMA_PATH):
        print('Store anterior eliminado.')
    else:
        print('[AVISO] No se pudo eliminar el directorio — continuando de todos modos.')

# Crear vector store persistente con los chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=CHROMA_PATH,
    collection_name='u5_05_knowledge'
)

print(f'Vector store creado: {vectorstore._collection.count()} vectores indexados')


c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\chromadb\types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


Vector store creado: 10 vectores indexados


In [8]:
# ============================================================
# Paso 5: Retrieval básico — k-NN semántico
# ============================================================
query = '¿Qué diferencia hay entre memoria episódica y semántica?'

# Recuperar los 3 chunks más similares
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
results = retriever.invoke(query)

print(f'Query: "{query}"')
print(f'\nTop-3 resultados recuperados:')
for i, doc in enumerate(results, 1):
    print(f'\n  Resultado {i} (fuente: {doc.metadata["source"]})')
    print(f'  {doc.page_content[:150]}...')

c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\chromadb\types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


Query: "¿Qué diferencia hay entre memoria episódica y semántica?"

Top-3 resultados recuperados:

  Resultado 1 (fuente: memoir_paper)
  La memoria episódica en agentes de IA permite retener eventos de sesiones
        anteriores. A diferencia de la memoria semántica (conocimiento factu...

  Resultado 2 (fuente: rag_survey)
  RAG (Retrieval-Augmented Generation) combina un retriever de similitud
        semántica con un generador LLM. El retriever busca en un vector store l...

  Resultado 3 (fuente: langchain_docs)
  . Soporta ciclos, condicionales, y checkpointing. Ideal para workflows
        complejos donde los agentes necesitan tomar decisiones iterativas....


In [9]:
from dotenv import load_dotenv
load_dotenv(override=True)  # recarga .env sin reiniciar el kernel

# ============================================================
# Paso 6: Construir la cadena RAG completa con LangChain
# ============================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Selección del LLM — prioridad: OpenRouter → Gemini → OpenAI → Ollama
if OPENROUTER_API_KEY:
    from langchain_openai import ChatOpenAI
    OPENROUTER_MODEL = os.environ.get('OPENROUTER_MODEL', 'google/gemini-2.5-flash')
    llm = ChatOpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=OPENROUTER_API_KEY,
        model=OPENROUTER_MODEL,
        temperature=0,
        default_headers={
            'HTTP-Referer': 'https://github.com/antigravity-nano',
            'X-Title': 'Antigravity Nano IA Course',
        },
    )
    print(f'LLM: OpenRouter — {OPENROUTER_MODEL}')
elif GOOGLE_API_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0)
    print('LLM: Gemini 2.0 Flash')
# elif OPENAI_API_KEY:
    # from langchain_openai import ChatOpenAI
    # llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, api_key=OPENAI_API_KEY)
    # print('LLM: GPT-4o-mini')
# else:
    # from langchain_ollama import ChatOllama
    # llm = ChatOllama(model='llama3.2', temperature=0)
    # print('LLM: Llama 3.2 via Ollama (local)')

# Prompt RAG estándar con contexto inyectado
rag_prompt = ChatPromptTemplate.from_template("""
Responde la siguiente pregunta usando únicamente la información del contexto proporcionado.
Si la información no está en el contexto, di explícitamente: "No tengo esa información en mi base de conocimiento."

Contexto:
{context}

Pregunta: {question}

Respuesta:
""")

def format_docs(docs):
    """Combina los chunks recuperados en un solo string para el contexto."""
    return '\n\n---\n\n'.join(doc.page_content for doc in docs)

# Pipeline RAG usando LCEL (LangChain Expression Language)
# La sintaxis | encadena: retriever → formato → prompt → LLM → parser
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Ejecutar el pipeline
response = rag_chain.invoke(query)
print(f'Pregunta: {query}')
print(f'\nRespuesta RAG:')
print(response)


LLM: OpenRouter — google/gemini-2.5-flash


c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\chromadb\types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


Pregunta: ¿Qué diferencia hay entre memoria episódica y semántica?

Respuesta RAG:
La memoria episódica tiene contexto temporal (qué ocurrió, cuándo, y en qué conversación), mientras que la memoria semántica es conocimiento factual.


**Output esperado:** La respuesta debe mencionar la diferencia temporal/contextual de la memoria episódica vs el conocimiento factual descontextualizado de la semántica, basándose en el chunk correspondiente del vector store.

**Interpretación:** El pipeline RAG recuperó los chunks semánticamente más cercanos a la query y los usó como contexto. El LLM no «inventó» información — solo sintetizó lo que proviene del vector store. Si la información no existiera en los documentos, la respuesta correcta es admitir la ignorancia (esto es lo que diferencia un RAG bien configurado de un LLM alucinando).

**Error común documentado:** Si en lugar de describir la diferencia, el LLM da una respuesta muy genérica, probablemente el chunking cortó el documento de memoria en un lugar inoportuno. Solución: aumentar `chunk_overlap` o cambiar a semantic chunking.

---
<a id='5-episodica'></a>
## 5. Memoria Episódica Vectorial

### 5.1 Mem0: Eventos como Vectores con Timestamps

Mem0 (pronunciado "mem-zero") es una librería de memoria persistente para agentes que unifica los 4 tipos de memoria en una API simple. La memoria episódica en Mem0 funciona así:

1. Al final de cada sesión, el sistema extrae los hechos importantes (`"el usuario prefiere respuestas cortas"`, `"el usuario trabaja en nanotecnología"`) usando un LLM de extracción
2. Esos hechos se convierten en embeddings y se almacenan con un timestamp
3. Al inicio de la siguiente sesión, se hace k-NN sobre esos hechos pasados relevantes a la query actual
4. Los hechos recuperados se inyectan en el system prompt como contexto episódico

**La diferencia con un vector store normal (RAG):** El RAG opera sobre documentos externos (PDFs, páginas web). La memoria episódica opera sobre el historial de **interacciones previas del usuario con el agente mismo**.

### 5.2 Skill `episodic_retriever`: Warm-Up con Episodios Pasados

In [10]:
# ============================================================
# WARM-UP: Skill episodic_retriever
# ============================================================
import sys
sys.path.insert(0, '../../../..')  # añadir raíz del proyecto al path

try:
    from external_skills.memory.episodic_retriever import (
        add_episode, retrieve, format_for_context, SKILL_METADATA
    )
    print('Skill episodic_retriever cargada desde external_skills/')
except ImportError:
    print('[AVISO] No se pudo importar desde external_skills/. Cargando inline...')
    # Definición inline mínima para que la notebook funcione de forma autónoma
    import json
    from pathlib import Path
    from datetime import datetime
    
    _store = {}
    
    def add_episode(content, user_id='default', metadata=None):
        ts = datetime.utcnow().isoformat()
        if user_id not in _store:
            _store[user_id] = []
        _store[user_id].append({'id': f'{user_id}_{ts}', 'content': content, 'timestamp': ts})
        return {'id': f'{user_id}_{ts}', 'timestamp': ts, 'backend': 'in_memory', 'success': True}
    
    def retrieve(query, user_id='default', top_k=5):
        q_words = set(query.lower().split())
        scored = []
        for ep in _store.get(user_id, []):
            overlap = len(q_words & set(ep['content'].lower().split()))
            if overlap > 0:
                scored.append({**ep, 'relevance_score': round(overlap / len(q_words), 2)})
        return sorted(scored, key=lambda x: x['relevance_score'], reverse=True)[:top_k]
    
    def format_for_context(episodes):
        if not episodes:
            return 'No hay episodios relevantes previos.'
        return '\n'.join(f'- {e["content"]} [{e["timestamp"][:10]}]' for e in episodes)
    
    SKILL_METADATA = {'name': 'episodic_retriever_inline', 'version': '1.0.0-inline'}

print(f'Skill: {SKILL_METADATA["name"]} v{SKILL_METADATA["version"]}')

[AVISO] No se pudo importar desde external_skills/. Cargando inline...
Skill: episodic_retriever_inline v1.0.0-inline


In [11]:
# ============================================================
# Paso 1: Simular sesiones pasadas — almacenar episodios
# ============================================================
USER_ID = 'estudiante_u5'

# Episodios de sesiones anteriores del usuario
episodios_pasados = [
    'El usuario está desarrollando un agente de análisis de literatura científica sobre nanopartículas.',
    'El usuario prefiere respuestas concisas con ejemplos de código en Python.',
    'El usuario ya completó U5_01 y U5_02. No necesita explicaciones básicas de LangChain.',
    'El usuario tiene presupuesto limitado de API: máximo $5 por sesión.',
    'La última sesión fue sobre implementar un agente CrewAI para comparar papers de DFT.',
]

for episodio in episodios_pasados:
    result = add_episode(episodio, user_id=USER_ID)
    print(f'  Guardado [{result["backend"]}]: {episodio[:60]}...')

print(f'\nTotal de episodios almacenados para {USER_ID}: {len(episodios_pasados)}')

  Guardado [in_memory]: El usuario está desarrollando un agente de análisis de liter...
  Guardado [in_memory]: El usuario prefiere respuestas concisas con ejemplos de códi...
  Guardado [in_memory]: El usuario ya completó U5_01 y U5_02. No necesita explicacio...
  Guardado [in_memory]: El usuario tiene presupuesto limitado de API: máximo $5 por ...
  Guardado [in_memory]: La última sesión fue sobre implementar un agente CrewAI para...

Total de episodios almacenados para estudiante_u5: 5


In [12]:
# ============================================================
# Paso 2: Warm-up de nueva sesión — recuperar episodios relevantes
# ============================================================
# Simula el inicio de una nueva sesión: el usuario escribe su primera consulta
nueva_query = '¿Cómo implemento memoria en un agente de investigación de papers?'

# Recuperar episodios relevantes de sesiones anteriores
episodios_relevantes = retrieve(nueva_query, user_id=USER_ID, top_k=3)

print(f'Query de nueva sesión: "{nueva_query}"')
print(f'\nEpisodios relevantes recuperados: {len(episodios_relevantes)}')
for ep in episodios_relevantes:
    print(f'  [relevancia: {ep["relevance_score"]}] {ep["content"][:80]}...')

# Formatear para inyectar en el system prompt del agente
contexto_episodico = format_for_context(episodios_relevantes)
print(f'\nContexto episódico para system prompt:')
print(contexto_episodico)

Query de nueva sesión: "¿Cómo implemento memoria en un agente de investigación de papers?"

Episodios relevantes recuperados: 3
  [relevancia: 0.33] El usuario está desarrollando un agente de análisis de literatura científica sob...
  [relevancia: 0.33] La última sesión fue sobre implementar un agente CrewAI para comparar papers de ...
  [relevancia: 0.22] El usuario prefiere respuestas concisas con ejemplos de código en Python....

Contexto episódico para system prompt:
- El usuario está desarrollando un agente de análisis de literatura científica sobre nanopartículas. [2026-04-02]
- La última sesión fue sobre implementar un agente CrewAI para comparar papers de DFT. [2026-04-02]
- El usuario prefiere respuestas concisas con ejemplos de código en Python. [2026-04-02]


In [13]:
# ============================================================
# Paso 3: Agente con warm-up episódico integrado
# ============================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# El system prompt incluye el contexto episódico recuperado
# Esto hace que el agente "recuerde" quién es el usuario sin que este lo repita
system_with_memory = f"""Eres un asistente de investigación científica experto en sistemas multi-agente.

Lo que recuerdas de sesiones anteriores con este usuario:
{contexto_episodico}

Usa esta información para personalizar tu respuesta.
No menciones explícitamente que tienes memoria de sesiones anteriores."""

agent_prompt = ChatPromptTemplate.from_messages([
    ('system', system_with_memory),
    ('human', '{question}')
])

agent_with_memory = agent_prompt | llm | StrOutputParser()

# Respuesta con contexto episódico
respuesta = agent_with_memory.invoke({'question': nueva_query})
print(f'Pregunta: {nueva_query}')
print(f'\nRespuesta (con memoria episódica):')
print(respuesta[:600], '...' if len(respuesta) > 600 else '')

Pregunta: ¿Cómo implemento memoria en un agente de investigación de papers?

Respuesta (con memoria episódica):
Implementar memoria en un agente de investigación de papers es crucial para que pueda aprender y mejorar con el tiempo. Aquí te presento algunas estrategias comunes, con ejemplos en Python, enfocadas en tu agente de análisis de literatura científica sobre nanopartículas.

### 1. Memoria a Corto Plazo (Contexto de la Conversación)

Para mantener el contexto de la interacción actual, puedes usar la memoria integrada en las herramientas de orquestación de agentes.

**Ejemplo con CrewAI (o similar):**

CrewAI maneja la memoria de la conversación automáticamente dentro de cada `Task` y `Agent` si s ...


**Output esperado:** La respuesta debe personalizar el consejo según el contexto episódico — por ejemplo, omitir explicaciones básicas de LangChain (porque el usuario ya completó U5_02), usar Python en ejemplos, y ser concisa.

**Interpretación:** El agente no "recuerda" por magia — simplemente recibe el contexto episódico relevante en su system prompt. Lo poderoso es que esta recuperación es automática y escala: aunque haya 500 episodios almacenados, el retriever selecciona los 3 más relevantes para esta query específica.

**Diferencia con RAG:** En RAG recuperamos chunks de documentos externos. Aquí recuperamos episodios de interacciones pasadas del usuario. Son el mismo mecanismo técnico (embeddings + k-NN) pero con fuentes de datos completamente diferentes y propósitos distintos.

### 5.3 Long-Term Memory Semántica con ChromaDB

El vector store de la Sección 4 (con documentos sobre frameworks) es la memoria semántica del agente. Para hacerla persistente y actualizable, solo hace falta configurar `persist_directory` en ChromaDB (ya lo hicimos en el Paso 4).

En producción, el vector store se actualiza cuando:
- Llegan documentos nuevos (crawling, uploads)
- Una respuesta del agente fue marcada como incorrecta (feedback loop)
- El usuario añade documentos personalizados

---
<a id='6-tokens'></a>
## 6. Optimización de Tokens y Presupuesto

### 6.1 El Problema: Costo = Riesgo Real

Un sistema multi-agente sin control de presupuesto puede generar facturas inesperadas. Un loop de agente que falla 20 veces antes de resolverse, con un LLM de $10/1M tokens y 2000 tokens por llamada, cuesta $0.40 — en *un solo error*. A escala de producción con cientos de usuarios, esto se vuelve un riesgo operativo real.

Las tres estrategias principales de optimización:

**1. ConversationSummaryMemory** — en lugar de pasar todo el historial, pasar un resumen:
```
Buffer (1000 tokens) → LLM resumen → Summary (200 tokens)
```

**2. Selective Context Loading** — no cargar todo el vector store en contexto, solo los chunks top-k más relevantes (ya lo hacemos con k=3 en la Sección 4).

**3. Semantic Caching** — si la misma query (o una semánticamente equivalente) ya fue procesada, devolver la respuesta cacheada sin llamar al LLM.

### 6.2 ConversationSummaryMemory

In [14]:
# ============================================================
# Paso 1: ConversationSummaryMemory — comprime el historial largo
# ============================================================
from langchain.memory import ConversationSummaryMemory

# Comparar: Buffer vs Summary
buffer_mem = ConversationBufferMemory(return_messages=True)
summary_mem = ConversationSummaryMemory(llm=llm, return_messages=True)

# Simular una conversación larga
conversacion = [
    ('¿Qué es LangGraph?', 'LangGraph es un framework para workflows multi-agente con estado y ciclos.'),
    ('¿Y CrewAI?', 'CrewAI es un framework role-based donde cada agente tiene un rol específico.'),
    ('¿Cuál es mejor para mi caso?', 'Depende: LangGraph si necesitas control de flujo complejo; CrewAI si prefieres roles.'),
    ('¿Puedo combinarlos?', 'Sí, puedes usar LangGraph para orquestar crews de CrewAI como nodos del grafo.'),
    ('¿Cómo manejo la memoria en LangGraph?', 'Usa MemorySaver o SqliteSaver como checkpointer en el StateGraph.'),
]

for human, ai in conversacion:
    buffer_mem.save_context({'input': human}, {'output': ai})
    summary_mem.save_context({'input': human}, {'output': ai})

print('=== BUFFER MEMORY ===')
buffer_msgs = buffer_mem.chat_memory.messages
buffer_tokens = sum(len(m.content.split()) * 1.3 for m in buffer_msgs)  # aprox tokens
print(f'Mensajes: {len(buffer_msgs)} | Tokens aprox: {int(buffer_tokens)}')

print('\n=== SUMMARY MEMORY ===')
print(f'Resumen actual:')
print(summary_mem.buffer[:300])
summary_tokens = len(summary_mem.buffer.split()) * 1.3
print(f'\nTokens aprox del resumen: {int(summary_tokens)}')
print(f'Ahorro: {int((1 - summary_tokens/buffer_tokens) * 100)}%')

C:\Users\UCEMICH\AppData\Local\Temp\ipykernel_14696\484240990.py:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  summary_mem = ConversationSummaryMemory(llm=llm, return_messages=True)


=== BUFFER MEMORY ===
Mensajes: 10 | Tokens aprox: 98

=== SUMMARY MEMORY ===
Resumen actual:
Current summary:
The human asks what LangGraph is, and the AI explains it's a framework for multi-agent workflows with state and cycles. The human then asks about CrewAI, and the AI describes it as a role-based framework where each agent has a specific role. The human then asks which is better for t

Tokens aprox del resumen: 159
Ahorro: -61%


**Output esperado:** El summary debe contener la esencia de la conversación en ~50-80 tokens vs ~200+ del buffer. Ahorro típico: 60-75%.

**Interpretación:** Para conversaciones cortas (<10 turnos), el buffer es más fiel. Para conversaciones largas (asistentes de soporte, tutores adaptativos), el summary es esencial para no agotar la ventana de contexto del LLM ni gastar tokens innecesarios.

### 6.3 Skill `token_budget_guard`: Circuit Breaker en Tiempo Real

In [15]:
# ============================================================
# WARM-UP: Skill token_budget_guard
# ============================================================
try:
    from external_skills.apis.token_budget_guard import (
        BudgetGuard, BudgetExceededError, estimate_tokens, SKILL_METADATA as TBG_META
    )
    print('Skill token_budget_guard cargada desde external_skills/')
except ImportError:
    print('[AVISO] Cargando token_budget_guard inline...')
    # Stub mínimo para ilustración
    class BudgetExceededError(RuntimeError): pass
    
    class BudgetGuard:
        MODEL_COSTS = {
            'gemini-2.0-flash': {'input': 0.10, 'output': 0.40},
            'gpt-4o': {'input': 2.50, 'output': 10.00},
            'ollama': {'input': 0.0, 'output': 0.0},
        }
        def __init__(self, budget_usd, model='gemini-2.0-flash'):
            self.budget_usd = budget_usd
            self.model = model
            self.cost_usd = 0.0
            self.tokens_input = 0
            self.tokens_output = 0
            self.circuit_open = False
        def record_call(self, input_tokens, output_tokens, desc=''):
            if self.circuit_open:
                raise BudgetExceededError('Presupuesto agotado.')
            costs = self.MODEL_COSTS.get(self.model, {'input': 0.10, 'output': 0.40})
            self.cost_usd += (input_tokens/1e6)*costs['input'] + (output_tokens/1e6)*costs['output']
            self.tokens_input += input_tokens
            self.tokens_output += output_tokens
            if self.cost_usd >= self.budget_usd:
                self.circuit_open = True
            return self.status()
        def status(self):
            return {'cost_usd': round(self.cost_usd, 6), 'budget_remaining_usd': round(self.budget_usd - self.cost_usd, 6), 'circuit_open': self.circuit_open}
        def report(self):
            return f'Costo: ${self.cost_usd:.6f} / ${self.budget_usd:.4f} | Circuit: {"ABIERTO" if self.circuit_open else "cerrado"}'
    
    def estimate_tokens(text, model='gpt-4o'):
        return int(len(text.split()) * 1.3)
    
    TBG_META = {'name': 'token_budget_guard_inline'}

print(f'Skill: {TBG_META["name"]}')

[AVISO] Cargando token_budget_guard inline...
Skill: token_budget_guard_inline


In [16]:
# ============================================================
# Paso 1: Configurar el guard y registrar llamadas
# ============================================================
# Presupuesto de $0.05 para esta sesión de demostración
guard = BudgetGuard(budget_usd=0.05, model='gemini-2.0-flash')

# Simular varias llamadas al LLM registrando tokens consumidos
llamadas_simuladas = [
    (500,  200,  'RAG retrieval query 1'),
    (800,  350,  'RAG retrieval query 2'),
    (1200, 500,  'Síntesis de documentos'),
    (600,  250,  'Memoria episódica warm-up'),
    (2000, 800,  'Generación de reporte final'),
]

print(f'Presupuesto configurado: ${guard.budget_usd:.4f} USD')
print(f'Modelo: {guard.model}\n')

for in_tokens, out_tokens, desc in llamadas_simuladas:
    try:
        status = guard.record_call(in_tokens, out_tokens, desc)
        print(f'  OK  | {desc:<35} | Costo acum: ${status["cost_usd"]:>10.6f} | Restante: ${status["budget_remaining_usd"]:>10.6f}')
    except BudgetExceededError as e:
        print(f'  BLOQUEADO | {desc:<35} | CIRCUIT BREAKER ACTIVADO')
        print(f'  Mensaje: {e}')
        break

print(f'\n{guard.report()}')

Presupuesto configurado: $0.0500 USD
Modelo: gemini-2.0-flash

  OK  | RAG retrieval query 1               | Costo acum: $  0.000130 | Restante: $  0.049870
  OK  | RAG retrieval query 2               | Costo acum: $  0.000350 | Restante: $  0.049650
  OK  | Síntesis de documentos              | Costo acum: $  0.000670 | Restante: $  0.049330
  OK  | Memoria episódica warm-up           | Costo acum: $  0.000830 | Restante: $  0.049170
  OK  | Generación de reporte final         | Costo acum: $  0.001350 | Restante: $  0.048650

Costo: $0.001350 / $0.0500 | Circuit: cerrado


**Output esperado:** Las primeras llamadas pasan, y en algún punto el circuit breaker se activa mostrando `BLOQUEADO`. Al llegar al presupuesto de $0.05, ninguna llamada posterior se procesa.

**Interpretación:** El `BudgetGuard` actúa como un `max_iterations` financiero. Sin él, un agente en loop indefinido podría gastar cientos de dólares antes de que el desarrollador lo note. Con él, el sistema falla de forma predecible y auditable.

**Para producción:** integrar `guard.record_call()` en el callback de LangSmith o en un middleware de LangChain que registre automáticamente los tokens de cada llamada.

---
<a id='7-apis'></a>
## 7. APIs de Datos como Fuentes de Memoria Semántica

Hasta ahora nuestro vector store contenía documentos estáticos. En sistemas de investigación reales, la memoria semántica debe alimentarse de fuentes dinámicas: nuevos papers, datos de mercado actualizados, noticias científicas.

El patrón es siempre el mismo:
```
API de datos → fetch() → chunks → embeddings → vector store
                              ↓
                    [agente consulta el vector store]
```

### 7.1 arXiv: Literatura Científica en Tiempo Real

In [17]:
# ============================================================
# Paso 1: Consultar arXiv API y cargar papers en el vector store
# ============================================================
import httpx
import xml.etree.ElementTree as ET

def fetch_arxiv_papers(query: str, max_results: int = 5) -> list[Document]:
    """
    Descarga los abstracts de papers de arXiv para una query dada.
    No requiere API key — arXiv es de acceso libre.
    """
    url = 'https://export.arxiv.org/api/query'  # https (arXiv redirige http → https)
    params = {
        'search_query': f'all:{query}',
        'start': 0,
        'max_results': max_results,
        'sortBy': 'relevance',
    }
    
    response = httpx.get(url, params=params, timeout=15)
    response.raise_for_status()
    
    root = ET.fromstring(response.text)
    ns = {'atom': 'http://www.w3.org/2005/Atom'}
    
    docs = []
    for entry in root.findall('atom:entry', ns):
        title = entry.find('atom:title', ns).text.strip()
        abstract = entry.find('atom:summary', ns).text.strip()
        published = entry.find('atom:published', ns).text[:10]
        arxiv_id = entry.find('atom:id', ns).text.strip().split('/')[-1]
        
        docs.append(Document(
            page_content=f"Título: {title}\n\nAbstract: {abstract}",
            metadata={'source': 'arxiv', 'arxiv_id': arxiv_id, 'date': published, 'topic': query}
        ))
    
    return docs

# Buscar papers sobre multi-agent systems
print('Consultando arXiv...')
arxiv_docs = fetch_arxiv_papers('multi-agent LLM systems', max_results=3)

print(f'Papers recuperados: {len(arxiv_docs)}')
for doc in arxiv_docs:
    print(f"\n  [{doc.metadata['date']}] {doc.page_content[:100]}...")


Consultando arXiv...


Papers recuperados: 3

  [2022-03-16] Título: A Survey of Multi-Agent Deep Reinforcement Learning with Communication

Abstract: Communicat...

  [2013-11-20] Título: A Methodology to Engineer and Validate Dynamic Multi-level Multi-agent Based Simulations

Ab...

  [2024-12-09] Título: Augmenting the action space with conventions to improve multi-agent cooperation in Hanabi

A...


In [18]:
# ============================================================
# Paso 2: Añadir papers al vector store existente
# ============================================================
# Chunking de los abstracts (ya son cortos, pero mantenemos el pipeline)
arxiv_chunks = splitter.split_documents(arxiv_docs)

# Añadir al vector store sin recrearlo — esto es memoria semántica dinámica
vectorstore.add_documents(arxiv_chunks)

print(f'Chunks de arXiv añadidos: {len(arxiv_chunks)}')
print(f'Total vectores en el store: {vectorstore._collection.count()}')

# Verificar que podemos recuperar papers de arXiv
query_arxiv = 'Latest research on multi-agent language model systems'
results = retriever.invoke(query_arxiv)

print(f'\nQuery: "{query_arxiv}"')
print(f'Fuentes recuperadas: {[r.metadata.get("source") for r in results]}')

c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\chromadb\types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


Chunks de arXiv añadidos: 20
Total vectores en el store: 30

Query: "Latest research on multi-agent language model systems"
Fuentes recuperadas: ['arxiv', 'arxiv', 'arxiv']


c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\chromadb\types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


**Output esperado:** El vector store ahora devuelve documentos de `arxiv` además de los documentos estáticos originales. La búsqueda semántica no distingue la fuente — recupera por relevancia.

**Interpretación:** Con este patrón, el agente tiene una memoria semántica que se actualiza automáticamente. Un cron job que ejecute `fetch_arxiv_papers()` cada día y añada los resultados al vector store daría al agente acceso perpetuo a la literatura científica más reciente — sin fine-tuning, sin reentrenamiento.

### 7.2 Patrón General: Cualquier API como Fuente RAG

El mismo patrón aplica a cualquier API:
- **PubMed**: `E-utilities API` → abstracts de biomedicina → chunks → vector store
- **Alpha Vantage**: `TIME_SERIES_DAILY` → datos de mercado → chunks → vector store
- **NASA APIs**: `APOD`, `Mars Rover Photos` → descripciones → vector store
- **GitHub**: `search/repositories` → READMEs → vector store

La única diferencia es el parser de respuesta de cada API. El resto del pipeline (chunking → embeddings → ChromaDB) es idéntico.

---
<a id='8-proyecto'></a>
## 8. Proyecto: Sistema de Investigación con los 4 Tipos de Memoria

Integramos todo lo visto en un agente de investigación completo que usa los 4 tipos de memoria simultáneamente y está controlado por un presupuesto de tokens.

In [19]:
# ============================================================
# WARM-UP: Sistema de Investigación — los 4 tipos de memoria
# ============================================================
from langchain.tools import Tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationSummaryMemory

print('Importaciones del sistema de investigación cargadas.')

Importaciones del sistema de investigación cargadas.


In [20]:
# ============================================================
# Paso 1: Configurar los 4 tipos de memoria
# ============================================================

# [1] PROCEDIMENTAL — cargar skills del registry
# La skill que usamos es el retriever del vector store y el episodic_retriever
rag_tool = Tool(
    name='search_knowledge_base',
    func=lambda q: '\n\n'.join(doc.page_content for doc in retriever.invoke(q)),
    description='Busca en la base de conocimiento sobre sistemas multi-agente y papers de arXiv. Úsala cuando necesites información técnica o científica.'
)

def buscar_episodios(query: str) -> str:
    """Tool que recupera episodios relevantes de sesiones pasadas."""
    eps = retrieve(query, user_id=USER_ID, top_k=3)
    return format_for_context(eps)

episodic_tool = Tool(
    name='recall_past_sessions',
    func=buscar_episodios,
    description='Recupera información de sesiones anteriores del usuario. Úsala cuando necesites recordar preferencias o trabajo previo del usuario.'
)

tools = [rag_tool, episodic_tool]

# [2] EPISÓDICA — contexto de sesiones pasadas (ya configurado en warm-up)
episodic_context = format_for_context(retrieve('investigación agentes multi-agente', user_id=USER_ID, top_k=2))

# [3] SEMÁNTICA — vector store listo (ya inicializado con docs + arXiv)

# [4] SENSORIAL — ConversationSummaryMemory para la sesión actual
session_memory = ConversationSummaryMemory(
    llm=llm,
    memory_key='chat_history',
    return_messages=True
)

print('Los 4 tipos de memoria configurados:')
print('  [1] Procedimental: 2 skills (rag_tool, episodic_tool)')
print('  [2] Episódica: Mem0/local episodic_retriever activo')
print('  [3] Semántica: ChromaDB con docs + arXiv')
print('  [4] Sensorial: ConversationSummaryMemory activa')

Los 4 tipos de memoria configurados:
  [1] Procedimental: 2 skills (rag_tool, episodic_tool)
  [2] Episódica: Mem0/local episodic_retriever activo
  [3] Semántica: ChromaDB con docs + arXiv
  [4] Sensorial: ConversationSummaryMemory activa


In [21]:
# ============================================================
# Paso 2: Construir el agente con presupuesto de tokens
# ============================================================

# Circuit breaker: $0.05 máximo para esta sesión de demo
session_guard = BudgetGuard(budget_usd=0.05, model='gemini-2.0-flash' if not USE_LOCAL else 'ollama')

agent_system_prompt = f"""Eres un asistente de investigación científica experto en sistemas multi-agente e IA.

Contexto de sesiones anteriores con este usuario:
{episodic_context}

Herramientas disponibles:
- search_knowledge_base: Para buscar información técnica y papers
- recall_past_sessions: Para recuperar información de sesiones anteriores

Instrucciones:
1. Usa search_knowledge_base para preguntas técnicas antes de responder desde tu memoria interna
2. Usa recall_past_sessions si el usuario hace referencia a trabajo previo
3. Sé conciso y directo — el usuario es un experto (U5_02 completada)
4. Siempre incluye la fuente cuando uses información del knowledge base"""

agent_prompt = ChatPromptTemplate.from_messages([
    ('system', agent_system_prompt),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=session_memory,
    max_iterations=5,       # límite de iteraciones (previene loops infinitos)
    verbose=False,
    handle_parsing_errors=True
)

print('Agente de investigación construido.')
print(f'Presupuesto de sesión: ${session_guard.budget_usd:.4f} USD')

Agente de investigación construido.
Presupuesto de sesión: $0.0500 USD


In [22]:
# ============================================================
# Paso 3: Sesión de investigación — múltiples consultas
# ============================================================

def ask_agent(question: str, guard: BudgetGuard) -> str:
    """Ejecuta una consulta al agente con control de presupuesto."""
    # Estimar tokens antes de llamar (input aprox)
    input_tokens_est = estimate_tokens(question + agent_system_prompt)
    
    try:
        # Verificar que aún tenemos presupuesto antes de llamar
        if guard.circuit_open:
            return '[BLOQUEADO] Presupuesto de sesión agotado. Circuit breaker activo.'
        
        response = agent_executor.invoke({'input': question})
        output = response['output']
        
        # Registrar tokens consumidos (estimación conservadora)
        output_tokens_est = estimate_tokens(output)
        guard.record_call(input_tokens_est, output_tokens_est, desc=question[:50])
        
        return output
    except BudgetExceededError:
        return '[BLOQUEADO] Presupuesto agotado durante la llamada.'
    except Exception as e:
        return f'[ERROR] {str(e)[:200]}'

# Consultas de investigación
consultas = [
    '¿Qué diferencia hay entre LangGraph y CrewAI según los documentos disponibles?',
    '¿Hay papers recientes relevantes para mi proyecto de agentes de investigación de nanopartículas?',
]

for i, consulta in enumerate(consultas, 1):
    print(f'\n--- Consulta {i} ---')
    print(f'Usuario: {consulta}')
    respuesta = ask_agent(consulta, session_guard)
    print(f'Agente: {respuesta[:400]}{"..." if len(respuesta) > 400 else ""}')
    print(f'[Presupuesto: ${session_guard.cost_usd:.5f} / ${session_guard.budget_usd:.4f}]')

print(f'\n{session_guard.report()}')


--- Consulta 1 ---
Usuario: ¿Qué diferencia hay entre LangGraph y CrewAI según los documentos disponibles?


c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\chromadb\types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


Agente: LangGraph es un framework que extiende LangChain para construir aplicaciones multi-agente con estado, utilizando grafos para definir nodos (funciones), aristas (transiciones) y estado compartido. Soporta ciclos, condicionales y checkpointing.

CrewAI es un framework basado en roles para sistemas multi-agente, donde cada agente tiene un rol, objetivo y trasfondo. Las tareas se asignan a agentes esp...
[Presupuesto: $0.00005 / $0.0500]

--- Consulta 2 ---
Usuario: ¿Hay papers recientes relevantes para mi proyecto de agentes de investigación de nanopartículas?


c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\chromadb\types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


Agente: No se encontraron papers recientes específicos sobre agentes de investigación de nanopartículas en la base de conocimiento. Sin embargo, hay información sobre metodologías de agentes que activan/desactivan o agregan/desagregan agentes para representar fenómenos a diferentes escalas, y sobre la comunicación como mecanismo efectivo para coordinar el comportamiento de múltiples agentes.

Si deseas, p...
[Presupuesto: $0.00010 / $0.0500]

Costo: $0.000104 / $0.0500 | Circuit: cerrado


In [23]:
# ============================================================
# Paso 4: Guardar la sesión como nuevo episodio (cierre de sesión)
# ============================================================
# Al final de cada sesión se guarda un episodio que estará disponible
# en el warm-up de la próxima sesión
resumen_sesion = session_memory.buffer if hasattr(session_memory, 'buffer') else 'Sesión completada'

nuevo_episodio = f"Sesión de investigación sobre RAG + memoria episódica. {resumen_sesion[:200]}"
result = add_episode(nuevo_episodio, user_id=USER_ID, metadata={'notebook': 'U5_05', 'session_cost_usd': session_guard.cost_usd})

print('Sesión guardada como episodio para la próxima sesión.')
print(f'Backend: {result["backend"]}')
print(f'Costo total de la sesión: ${session_guard.cost_usd:.5f} USD')

Sesión guardada como episodio para la próxima sesión.
Backend: in_memory
Costo total de la sesión: $0.00010 USD


**Output esperado:** Las consultas devuelven respuestas basadas en la base de conocimiento + contexto episódico. El presupuesto se decrementa con cada llamada. Al llegar al límite, el circuit breaker bloquea nuevas llamadas.

**Interpretación:** Este sistema demuestra la coordinación de los 4 tipos de memoria:
- **Procedimental**: Las tools `search_knowledge_base` y `recall_past_sessions` son las skills del agente
- **Episódica**: El contexto de sesiones anteriores está inyectado en el system prompt y disponible como tool
- **Semántica**: ChromaDB con documentos + papers de arXiv
- **Sensorial**: `ConversationSummaryMemory` mantiene el hilo de la sesión actual comprimido

El `BudgetGuard` hace que el sistema sea seguro para producción: falla de forma predecible y auditable en lugar de consumir tokens ilimitados.

---
<a id='9-resumen'></a>
## 9. Resumen y Conexión con Otras Notebooks

### Lo que construimos en U5_05

| Componente | Tecnología | Tipo de memoria |
|-----------|-----------|----------------|
| Pipeline RAG (chunking + embeddings + ChromaDB) | LangChain + HuggingFace | Semántica |
| Recuperación de episodios pasados | `episodic_retriever` skill + Mem0 | Episódica |
| Historial comprimido de sesión | `ConversationSummaryMemory` | Sensorial |
| Skills como tools del agente | Skill Registry + `create_tool_calling_agent` | Procedimental |
| Control de presupuesto | `token_budget_guard` con circuit breaker | Infraestructura |
| arXiv como fuente dinámica | httpx + ChromaDB `.add_documents()` | Semántica dinámica |

### Conexión con U5_06 (Graph RAG)

> **SCOPE BOUNDARY**: Esta notebook almacena memoria episódica como vectores + timestamps (Mem0/local). U5_06 almacena la misma categoría de información como **grafo temporal estructurado** (nodos = entidades, aristas = relaciones). Son enfoques complementarios: esta notebook para texto y eventos planos; U5_06 para relaciones complejas entre entidades que requieren traversal multi-hop.

### Conexión con U5_06

U5_06 extiende estos patterns a:
- APIs multimodales (imágenes, audio) como fuentes del vector store
- Exponer el sistema de investigación como API REST con FastAPI
- Model routing: elegir el modelo según presupuesto restante y complejidad
- `github_skill_loader`: cargar skills adicionales desde GitHub en producción

---
<a id='10-ejercicios'></a>
## 10. Ejercicios de Extensión

**Ejercicio 1 — Semantic Chunking (★★★☆☆)**  
Reemplaza `RecursiveCharacterTextSplitter` por `SemanticChunker` de LangChain experimental. Compara la calidad de recuperación en 5 queries distintas y documenta en qué casos el semantic chunking da mejores resultados.

```python
from langchain_experimental.text_splitter import SemanticChunker
semantic_splitter = SemanticChunker(embeddings_model)
# Tu código aquí
```

**Ejercicio 2 — Reranking con Cross-Encoder (★★★★☆)**  
Añade reranking al pipeline RAG usando `cross-encoder/ms-marco-MiniLM-L-6-v2` de sentence-transformers. Recupera k=20 con k-NN y reordena para quedarte con los top-5. Compara las respuestas con y sin reranking en queries ambiguas.

```python
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# Tu código aquí
```

**Ejercicio 3 — Memoria Episódica con Dominio Diferente (★★☆☆☆)**  
Adapta el sistema de investigación para un caso de uso diferente: un asistente de soporte técnico que recuerda los problemas reportados por cada usuario. Cambia `USER_ID` y los episodios para reflejar tickets de soporte.

**Ejercicio 4 — PubMed como Fuente RAG (★★★☆☆)**  
Implementa `fetch_pubmed_papers(query, max_results)` usando la E-utilities API de NCBI (sin API key, acceso libre). Añade los abstracts al vector store y demuestra que el agente puede responder preguntas sobre nanomedicina.

```python
BASE_URL = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/'
# Paso 1: esearch.fcgi para obtener IDs
# Paso 2: efetch.fcgi para obtener abstracts con rettype=abstract
# Tu código aquí
```

**Ejercicio 5 — Budget Guard en LangGraph (★★★★☆)**  
Integra `BudgetGuard` como nodo en un StateGraph de LangGraph. El grafo debe incluir:
- Nodo `check_budget`: evalúa si hay presupuesto antes de cada llamada LLM
- Arista condicional: si `circuit_open == True` → ir a nodo `graceful_exit`; si no → continuar al nodo `execute`
- Nodo `graceful_exit`: retorna un mensaje explicativo al usuario en lugar de fallar

In [24]:
# ============================================================
# Limpieza final — eliminar vector store persistente del disco
# ============================================================
if os.path.exists(CHROMA_PATH):
    if _close_and_rmtree(CHROMA_PATH):
        print(f'Vector store temporal eliminado: {CHROMA_PATH}')
    else:
        print(f'[AVISO] No se pudo eliminar {CHROMA_PATH} — cierra el notebook y bórralo manualmente.')

print('\nNotebook completada.')
print('Siguiente: U5_06 (Graph RAG y Memoria en Grafo) -> U5_07 (Multimodal y Producción)')


[AVISO] No se pudo eliminar ./chroma_u5_05 — cierra el notebook y bórralo manualmente.

Notebook completada.
Siguiente: U5_06 (Graph RAG y Memoria en Grafo) -> U5_07 (Multimodal y Producción)
